# Week 9-10: Full Experiments with Real Data

This notebook runs comprehensive CCE experiments on real repositories:

**Experiments:**
1. CCE Adaptive vs All Baselines (7 methods)
2. Measurement Strategy Comparison
3. Threshold Sensitivity Analysis
4. Ablation Study

**Data:**
- Flask, FastAPI, Requests repositories
- RepoSynth semantic packs with FAISS indices

---

## Setup Instructions

1. Upload this notebook to Colab
2. Enable GPU: Runtime -> Change runtime type -> T4 GPU
3. Run cells 1-3 to install deps and create structure
4. Upload module files (5 batches as in Week 8)
5. Upload pack files (vectors.faiss, vector_ids.json, name_registry.json per repo)
6. Run experiments

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn sentence-transformers faiss-cpu

In [ ]:
# Cell 2: Create directory structure
import os
import shutil

# Create module structure
os.makedirs('/content/orchestrator/entropy', exist_ok=True)
os.makedirs('/content/orchestrator/retrieval', exist_ok=True)
os.makedirs('/content/orchestrator/generation', exist_ok=True)
os.makedirs('/content/orchestrator/evaluation', exist_ok=True)
os.makedirs('/content/benchmarks', exist_ok=True)

# Create pack directories for each repo
os.makedirs('/content/packs/flask_pack', exist_ok=True)
os.makedirs('/content/packs/fastapi_pack', exist_ok=True)
os.makedirs('/content/packs/requests_pack', exist_ok=True)

# Results directory
os.makedirs('/content/results/figures', exist_ok=True)
os.makedirs('/content/results/tables', exist_ok=True)

# Create root __init__.py
with open('/content/orchestrator/__init__.py', 'w') as f:
    f.write('"""Orchestrator package."""\n')

print("Directory structure created:")
print("  /content/orchestrator/   (modules)")
print("  /content/packs/          (repository packs)")
print("  /content/benchmarks/     (evaluation data)")
print("  /content/results/        (output)")

In [ ]:
# Cell 3: Upload module files (same as Week 8)
from google.colab import files

def upload_to_dir(target_dir):
    """Upload files and move to target directory."""
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.py') or filename.endswith('.json'):
            dest = f'{target_dir}/{filename}'
            shutil.move(filename, dest)
            print(f"  -> {filename}")
    return uploaded

# ============================================================
print("="*60)
print("STEP 1/5: Upload ENTROPY module files (7 files)")
print("="*60)
print("\nFrom: packages/python-orchestrator/orchestrator/entropy/")
print("Upload: __init__.py, calculator.py, cce_computer.py, token_classifier.py,")
print("        measurement.py, monitor.py, spike_detector.py")
upload_to_dir('/content/orchestrator/entropy')

# ============================================================
print("\n" + "="*60)
print("STEP 2/5: Upload RETRIEVAL module files (4 files)")
print("="*60)
print("\nFrom: packages/python-orchestrator/orchestrator/retrieval/")
print("Upload: __init__.py, topic_inference.py, adaptive.py, context_manager.py")
upload_to_dir('/content/orchestrator/retrieval')

# ============================================================
print("\n" + "="*60)
print("STEP 3/5: Upload GENERATION module files (2 files)")
print("="*60)
print("\nFrom: packages/python-orchestrator/orchestrator/generation/")
print("Upload: __init__.py, adaptive_generator.py")
upload_to_dir('/content/orchestrator/generation')

# ============================================================
print("\n" + "="*60)
print("STEP 4/5: Upload EVALUATION module files (5 files)")
print("="*60)
print("\nFrom: packages/python-orchestrator/orchestrator/evaluation/")
print("Upload: __init__.py, benchmark.py, metrics.py, runner.py, stats.py")
upload_to_dir('/content/orchestrator/evaluation')

# ============================================================
print("\n" + "="*60)
print("STEP 5/5: Upload BENCHMARK file (1 file)")
print("="*60)
print("\nFrom: research/benchmarks/")
print("Upload: benchmark_v1.json")
upload_to_dir('/content/benchmarks')

print("\n" + "="*60)
print("VERIFICATION")
print("="*60)
!ls /content/orchestrator/evaluation/

In [ ]:
# Cell 4: Upload pack files for each repository
from google.colab import files

def upload_pack(pack_name, target_dir):
    """Upload pack files (FAISS index, vector IDs, name registry)."""
    print(f"\nUpload files for {pack_name}:")
    print(f"  - vectors.faiss")
    print(f"  - vector_ids.json")
    print(f"  - name_registry.json")
    print(f"  - repoBrief.md (optional)")
    
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest = f'{target_dir}/{filename}'
        shutil.move(filename, dest)
        print(f"  -> {filename}")
    return uploaded

# ============================================================
print("="*60)
print("UPLOAD REPOSITORY PACKS")
print("="*60)
print("\nFrom: research/packs/<repo>_pack/")
print("Upload vectors.faiss, vector_ids.json, name_registry.json for each repo")

print("\n--- FLASK PACK ---")
upload_pack('flask', '/content/packs/flask_pack')

print("\n--- FASTAPI PACK ---")
upload_pack('fastapi', '/content/packs/fastapi_pack')

print("\n--- REQUESTS PACK ---")
upload_pack('requests', '/content/packs/requests_pack')

# Verify
print("\n" + "="*60)
print("VERIFICATION")
print("="*60)
for pack in ['flask_pack', 'fastapi_pack', 'requests_pack']:
    print(f"\n{pack}:")
    !ls /content/packs/{pack}/

In [ ]:
# Cell 5: Base imports
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Any, Protocol
from dataclasses import dataclass, field
import time
import faiss
import torch

sys.path.insert(0, '/content')

import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Base imports complete")

In [ ]:
# Cell 6: Import evaluation modules
from orchestrator.evaluation import (
    BenchmarkDataset,
    BenchmarkExample,
    EvaluationMetrics,
    EvaluationResult,
    StatisticalAnalysis,
    BaselineMethod,
)
from orchestrator.evaluation.benchmark import Difficulty, Category
from orchestrator.evaluation.runner import (
    BaselineRunner,
    ExperimentRunner,
    EvaluationRetriever,
    ExperimentConfig,
)

print("Evaluation modules imported successfully!")

In [ ]:
# Cell 7: Import CCE modules
from orchestrator.entropy import (
    EntropyCalculator,
    CCEComputer,
    TokenClassifier,
    MeasurementStrategy,
    UncertaintyMonitor,
    SpikeDetector,
)
from orchestrator.retrieval import (
    TopicInference,
    AdaptiveRetriever,
    ContextManager,
)
from orchestrator.retrieval.adaptive import SimpleRetriever, AdaptiveContextRetriever
from orchestrator.retrieval.topic_inference import TopicInferrer
from orchestrator.generation import AdaptiveGenerator, GenerationConfig, GenerationResult

print("All CCE modules imported successfully!")

---
## 1. Load Real Repository Packs

In [ ]:
# Cell 8: Load repository packs and create FAISS-based retriever
from sentence_transformers import SentenceTransformer

class FAISSPackRetriever:
    """
    Retriever using RepoSynth packs with FAISS semantic search.
    
    Implements the BaseRetriever protocol expected by AdaptiveContextRetriever:
        retrieve(query: str, top_k: int) -> List[Dict[str, Any]]
    """
    
    def __init__(self, pack_dirs: Dict[str, str], embedding_model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize retriever with multiple repository packs.
        
        Args:
            pack_dirs: Dict mapping repo name to pack directory path
            embedding_model_name: SentenceTransformer model name
        """
        print(f"Loading embedding model: {embedding_model_name}")
        self.embedding_model = SentenceTransformer(embedding_model_name)
        
        # Load all packs
        self.indices = {}       # repo_name -> FAISS index
        self.vector_ids = {}    # repo_name -> {vector_id: symbol_id}
        self.registries = {}    # repo_name -> name_registry dict
        self.documents = {}     # file_key -> content (combined from all repos)
        
        for repo_name, pack_dir in pack_dirs.items():
            pack_path = Path(pack_dir)
            
            # Check required files exist
            faiss_path = pack_path / 'vectors.faiss'
            ids_path = pack_path / 'vector_ids.json'
            registry_path = pack_path / 'name_registry.json'
            
            if not all(p.exists() for p in [faiss_path, ids_path, registry_path]):
                print(f"  Skipping {repo_name}: missing required files")
                continue
            
            # Load FAISS index
            self.indices[repo_name] = faiss.read_index(str(faiss_path))
            
            # Load vector IDs mapping
            with open(ids_path) as f:
                self.vector_ids[repo_name] = json.load(f)
            
            # Load name registry
            with open(registry_path) as f:
                self.registries[repo_name] = json.load(f)
            
            # Build document store from registry
            for symbol_id, symbol in self.registries[repo_name].items():
                file_path = symbol.get('file', '')
                code = symbol.get('code', symbol.get('signature', ''))
                if file_path and code:
                    doc_key = f"{repo_name}/{file_path}"
                    if doc_key not in self.documents:
                        self.documents[doc_key] = ""
                    self.documents[doc_key] += f"\n# {symbol.get('name', symbol_id)}\n{code}\n"
            
            print(f"  Loaded {repo_name}: {self.indices[repo_name].ntotal} vectors, {len(self.registries[repo_name])} symbols")
        
        print(f"\nTotal documents: {len(self.documents)}")
    
    def retrieve(self, query: str, top_k: int = 3, repo: str = None, **kwargs) -> List[Dict[str, Any]]:
        """
        Retrieve relevant code using semantic search.
        
        Args:
            query: Search query
            top_k: Number of results to return
            repo: Optional specific repository to search
        
        Returns:
            List of dicts with 'source', 'content', 'score' keys
        """
        if not self.indices:
            return []
        
        # Encode query
        query_vec = self.embedding_model.encode([query]).astype('float32')
        
        all_results = []
        search_repos = {repo: self.indices[repo]} if repo and repo in self.indices else self.indices
        
        for repo_name, index in search_repos.items():
            # Search FAISS index
            distances, indices = index.search(query_vec, top_k)
            
            for dist, idx in zip(distances[0], indices[0]):
                if idx < 0:  # FAISS returns -1 for missing
                    continue
                
                # Map vector index to symbol
                symbol_id = self.vector_ids[repo_name].get(str(idx))
                if not symbol_id:
                    continue
                
                symbol = self.registries[repo_name].get(symbol_id, {})
                file_path = symbol.get('file', '')
                code = symbol.get('code', symbol.get('signature', ''))
                
                if file_path and code:
                    # Convert distance to similarity score (0-1)
                    # FAISS L2 distance: lower is better, convert to similarity
                    score = 1.0 / (1.0 + float(dist))
                    
                    all_results.append({
                        'source': f"{repo_name}/{file_path}",
                        'content': f"# {symbol.get('name', symbol_id)}\n{code}",
                        'score': score,
                        'symbol': symbol.get('name', symbol_id),
                        'type': symbol.get('type', 'unknown'),
                    })
        
        # Sort by score (descending) and return top_k
        all_results.sort(key=lambda x: x['score'], reverse=True)
        return all_results[:top_k]
    
    def retrieve_all(self) -> List[Tuple[str, str]]:
        """Return all documents (for full_context baseline)."""
        return list(self.documents.items())
    
    def get_document(self, key: str) -> Optional[str]:
        """Get a specific document by key."""
        return self.documents.get(key)


# Initialize retriever with available packs
pack_dirs = {
    'flask': '/content/packs/flask_pack',
    'fastapi': '/content/packs/fastapi_pack',
    'requests': '/content/packs/requests_pack',
}

# Filter to only existing packs
available_packs = {k: v for k, v in pack_dirs.items() if Path(v).exists() and (Path(v) / 'vectors.faiss').exists()}

if available_packs:
    base_retriever = FAISSPackRetriever(available_packs)
    print(f"\nRetriever initialized with {len(available_packs)} packs")
else:
    print("No packs available! Please upload pack files first.")
    base_retriever = None

In [ ]:
# Cell 9: Test semantic search on real packs
if base_retriever:
    print("Testing Semantic Search")
    print("="*60)
    
    test_queries = [
        "How does Flask handle URL routing?",
        "How does FastAPI handle dependency injection?",
        "How does requests handle HTTP sessions?",
    ]
    
    for query in test_queries:
        results = base_retriever.retrieve(query, top_k=3)
        
        print(f"\nQuery: {query}")
        for i, r in enumerate(results, 1):
            print(f"  {i}. [{r['score']:.3f}] {r['symbol']} ({r['type']}) - {r['source']}")
else:
    print("Skipping test - no retriever available")

---
## 2. Load Model and Setup

In [ ]:
# Cell 10: Load language model
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

print(f"Model loaded on {model.device}")
print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 11: Setup CCE Adaptive Generator with real retriever
print("Setting up CCE Adaptive Generator...")

# Configure generation
gen_config = GenerationConfig(
    max_tokens=150,
    max_retrievals=5,
    uncertainty_method="raw_entropy",  # Options: raw_entropy, cce
    uncertainty_threshold=2.5,
    measurement_strategy="line_boundary",  # Options: every_token, every_n_tokens, line_boundary, semantic_boundary
    cooldown_tokens=10,
    min_topic_confidence=0.3,
    min_relevance_score=0.1,
    top_k_retrieval=3,
    context_budget=4096,
)

# Create adaptive generator
if base_retriever:
    adaptive_generator = AdaptiveGenerator(
        model=model,
        tokenizer=tokenizer,
        retriever=base_retriever,
        config=gen_config,
    )
    print("CCE Adaptive Generator ready!")
    print(f"  - Threshold: {gen_config.uncertainty_threshold}")
    print(f"  - Strategy: {gen_config.measurement_strategy}")
    print(f"  - Max retrievals: {gen_config.max_retrievals}")
else:
    adaptive_generator = None
    print("Cannot create generator - no retriever available")

In [ ]:
# Cell 12: Load benchmark and setup metrics
benchmark = BenchmarkDataset.load('/content/benchmarks/benchmark_v1.json')

print("Benchmark Dataset")
print("="*60)
print(f"Total examples: {len(benchmark)}")

stats = benchmark.get_statistics()
print(f"By difficulty: {stats['by_difficulty']}")
print(f"By category: {stats['by_category']}")

# Filter to only repos we have packs for
if base_retriever:
    available_repos = set(base_retriever.indices.keys())
    filtered_examples = [ex for ex in benchmark if ex.repository in available_repos]
    benchmark_filtered = BenchmarkDataset(
        examples=filtered_examples,
        name="filtered_benchmark",
    )
    print(f"\nFiltered to {len(benchmark_filtered)} examples (repos: {available_repos})")
else:
    benchmark_filtered = benchmark
    print("\nUsing full benchmark (no filtering)")

# Initialize metrics
metrics = EvaluationMetrics(embedding_model='all-MiniLM-L6-v2')
print("\nEvaluationMetrics initialized")

---
## 3. Experiment 1: CCE Adaptive vs All Baselines

In [ ]:
# Cell 13: Define all experiment methods

def run_cce_adaptive(example: BenchmarkExample, generator: AdaptiveGenerator, max_tokens: int) -> Dict:
    """
    Run CCE adaptive generation with uncertainty-triggered retrieval.
    """
    result: GenerationResult = generator.generate(
        query=example.query,
        initial_context="",
    )
    
    return {
        'answer': result.response,
        'retrieved_files': [e.result.source for e in result.retrieval_events if e.result],
        'tokens_used': result.total_tokens + result.total_context_tokens,
        'num_retrievals': result.total_retrievals,
        'generation_time': result.generation_time,
        'spike_positions': [e.position for e in result.retrieval_events],
        'entropy_trace': result.entropy_trace,
    }


def run_no_context(example: BenchmarkExample, model, tokenizer, max_tokens: int) -> Dict:
    """
    Generate without any context retrieval.
    """
    start = time.time()
    
    inputs = tokenizer(example.query, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_tokens, pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return {
        'answer': answer,
        'retrieved_files': [],
        'tokens_used': len(tokenizer.encode(answer)),
        'num_retrievals': 0,
        'generation_time': time.time() - start,
        'spike_positions': [],
    }


def run_with_context(example: BenchmarkExample, model, tokenizer, retriever, max_tokens: int, method: str) -> Dict:
    """
    Generate with pre-retrieved context (embedding, bm25, etc).
    """
    start = time.time()
    
    # Retrieve context
    if method == 'full_context':
        # Use all documents (truncated)
        all_docs = retriever.retrieve_all()[:20]
        context = "\n".join([f"# {k}\n{v[:300]}" for k, v in all_docs])
        retrieved_files = [k for k, v in all_docs]
    else:
        # Semantic search
        docs = retriever.retrieve(example.query, top_k=3)
        context = "\n\n".join([f"{d['content']}" for d in docs])
        retrieved_files = [d['source'] for d in docs]
    
    # Format prompt
    prompt = f"""Context:
{context[:3000]}

Question: {example.query}
Answer:"""
    
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_tokens, pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return {
        'answer': answer,
        'retrieved_files': retrieved_files,
        'tokens_used': len(tokenizer.encode(prompt + answer)),
        'num_retrievals': 1,
        'generation_time': time.time() - start,
        'spike_positions': [],
    }


METHODS = ['cce_adaptive', 'no_context', 'embedding', 'bm25', 'full_context', 'reposynth_base']

print("Experiment methods defined:")
for m in METHODS:
    print(f"  - {m}")

In [ ]:
# Cell 14: Run Experiment 1 - CCE vs All Baselines
print("="*70)
print("EXPERIMENT 1: CCE Adaptive vs All Baselines")
print("="*70)

MAX_TOKENS = 150
BASELINE_TOKENS = 5000  # Approximate full context tokens

results_exp1 = {method: [] for method in METHODS}

# Run on filtered benchmark (or subset for faster testing)
test_examples = list(benchmark_filtered)[:10]  # Limit for testing, remove [:10] for full run

for i, example in enumerate(test_examples):
    print(f"\n[{i+1}/{len(test_examples)}] {example.id}: {example.query[:50]}...")
    
    for method in METHODS:
        try:
            # Run method
            if method == 'cce_adaptive':
                if adaptive_generator:
                    output = run_cce_adaptive(example, adaptive_generator, MAX_TOKENS)
                else:
                    continue
            elif method == 'no_context':
                output = run_no_context(example, model, tokenizer, MAX_TOKENS)
            else:
                if base_retriever:
                    output = run_with_context(example, model, tokenizer, base_retriever, MAX_TOKENS, method)
                else:
                    continue
            
            # Evaluate
            eval_result = metrics.evaluate(
                example_id=example.id,
                generated_answer=output['answer'],
                ground_truth_answer=example.ground_truth_answer,
                retrieved_files=output['retrieved_files'],
                ground_truth_files=example.ground_truth_files,
                ground_truth_keywords=example.ground_truth_keywords,
                tokens_used=output['tokens_used'],
                baseline_tokens=BASELINE_TOKENS,
                generation_time=output['generation_time'],
                num_retrievals=output['num_retrievals'],
                detected_positions=output.get('spike_positions', []),
                ground_truth_positions=example.ground_truth_missing_positions,
                method=method,
            )
            results_exp1[method].append(eval_result)
            
            print(f"    {method}: correct={eval_result.answer_correctness:.2f}, eff={eval_result.token_efficiency:.2f}, time={output['generation_time']:.1f}s")
            
        except Exception as e:
            print(f"    {method}: ERROR - {str(e)[:50]}")

print("\n" + "="*70)
print("Experiment 1 Complete!")
print("="*70)

In [ ]:
# Cell 15: Aggregate Experiment 1 Results
print("\nExperiment 1 Results")
print("="*80)

exp1_aggregates = {}

for method, results in results_exp1.items():
    if not results:
        continue
    
    exp1_aggregates[method] = {
        'n': len(results),
        'answer_correctness': np.mean([r.answer_correctness for r in results]),
        'answer_completeness': np.mean([r.answer_completeness for r in results]),
        'hallucination_rate': np.mean([r.hallucination_rate for r in results]),
        'context_precision': np.mean([r.context_precision for r in results]),
        'context_recall': np.mean([r.context_recall for r in results]),
        'context_f1': np.mean([r.get_f1_context() for r in results]),
        'token_efficiency': np.mean([r.token_efficiency for r in results]),
        'composite_score': np.mean([r.get_composite_score() for r in results]),
        'avg_time': np.mean([r.generation_time for r in results]),
    }

# Create DataFrame
df_exp1 = pd.DataFrame(exp1_aggregates).T
df_exp1 = df_exp1.round(3)
print(df_exp1.to_string())

# Save to CSV
df_exp1.to_csv('/content/results/exp1_method_comparison.csv')
print("\nSaved to /content/results/exp1_method_comparison.csv")

In [ ]:
# Cell 16: Statistical Significance Testing
print("\nStatistical Significance: CCE Adaptive vs Baselines")
print("="*80)

stats_analyzer = StatisticalAnalysis(confidence=0.95)

cce_results = results_exp1.get('cce_adaptive', [])
if cce_results and len(cce_results) >= 3:
    comparisons = []
    
    for method, baseline_results in results_exp1.items():
        if method == 'cce_adaptive' or not baseline_results or len(baseline_results) < 3:
            continue
        
        # Compare answer correctness
        cce_scores = [r.answer_correctness for r in cce_results]
        baseline_scores = [r.answer_correctness for r in baseline_results]
        
        # Ensure same length
        min_len = min(len(cce_scores), len(baseline_scores))
        cce_scores = cce_scores[:min_len]
        baseline_scores = baseline_scores[:min_len]
        
        comparison = stats_analyzer.compare_methods(
            method_a='cce_adaptive',
            scores_a=cce_scores,
            method_b=method,
            scores_b=baseline_scores,
            metric='answer_correctness',
        )
        comparisons.append(comparison)
    
    if comparisons:
        print(stats_analyzer.significance_summary(comparisons))
else:
    print("Not enough CCE adaptive results for statistical comparison")

In [ ]:
# Cell 17: Plot Experiment 1 Results
if exp1_aggregates:
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    methods = list(exp1_aggregates.keys())
    colors = plt.cm.Set2(np.linspace(0, 1, len(methods)))
    
    # 1. Composite Score Bar Chart
    ax1 = axes[0, 0]
    scores = [exp1_aggregates[m]['composite_score'] for m in methods]
    bars = ax1.bar(range(len(methods)), scores, color=colors)
    ax1.set_xticks(range(len(methods)))
    ax1.set_xticklabels(methods, rotation=45, ha='right')
    ax1.set_ylabel('Composite Score')
    ax1.set_title('Method Comparison: Composite Score')
    ax1.set_ylim(0, 1)
    for bar, score in zip(bars, scores):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{score:.3f}', ha='center', fontsize=9)
    
    # 2. Quality vs Efficiency Scatter
    ax2 = axes[0, 1]
    for i, m in enumerate(methods):
        ax2.scatter(exp1_aggregates[m]['token_efficiency'],
                    exp1_aggregates[m]['answer_correctness'],
                    s=150, c=[colors[i]], label=m, edgecolors='black')
    ax2.set_xlabel('Token Efficiency')
    ax2.set_ylabel('Answer Correctness')
    ax2.set_title('Quality vs Efficiency Trade-off')
    ax2.legend(loc='lower right', fontsize=8)
    ax2.set_xlim(-0.1, 1.1)
    ax2.set_ylim(0, 1)
    
    # 3. Multi-metric comparison
    ax3 = axes[1, 0]
    radar_metrics = ['answer_correctness', 'answer_completeness', 'context_f1', 'token_efficiency']
    x = np.arange(len(radar_metrics))
    width = 0.8 / len(methods)
    for i, m in enumerate(methods):
        values = [exp1_aggregates[m][metric] for metric in radar_metrics]
        ax3.bar(x + i*width, values, width, label=m, color=colors[i])
    ax3.set_xticks(x + width * (len(methods)-1) / 2)
    ax3.set_xticklabels(['Correctness', 'Completeness', 'Context F1', 'Efficiency'], rotation=45, ha='right')
    ax3.set_ylabel('Score')
    ax3.set_title('Multi-Metric Comparison')
    ax3.legend(loc='upper right', fontsize=7)
    ax3.set_ylim(0, 1)
    
    # 4. Hallucination Rate
    ax4 = axes[1, 1]
    halluc_rates = [exp1_aggregates[m]['hallucination_rate'] for m in methods]
    bars = ax4.bar(range(len(methods)), halluc_rates, color=colors)
    ax4.set_xticks(range(len(methods)))
    ax4.set_xticklabels(methods, rotation=45, ha='right')
    ax4.set_ylabel('Hallucination Rate')
    ax4.set_title('Hallucination Rate (Lower is Better)')
    ax4.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig('/content/results/figures/exp1_method_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Figure saved: /content/results/figures/exp1_method_comparison.png")
else:
    print("No results to plot")

---
## 4. Experiment 2: Threshold Sensitivity

In [ ]:
# Cell 18: Experiment 2 - Threshold Sensitivity
print("="*70)
print("EXPERIMENT 2: Threshold Sensitivity Analysis")
print("="*70)

THRESHOLDS = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
results_exp2 = {}

if adaptive_generator:
    test_examples_exp2 = list(benchmark_filtered)[:5]  # Subset for speed
    
    for threshold in THRESHOLDS:
        print(f"\nThreshold = {threshold}")
        results_exp2[threshold] = []
        
        # Reconfigure generator with new threshold
        adaptive_generator.config.uncertainty_threshold = threshold
        adaptive_generator._init_monitor()  # Reinitialize monitor with new threshold
        
        for example in test_examples_exp2:
            try:
                output = run_cce_adaptive(example, adaptive_generator, MAX_TOKENS)
                
                eval_result = metrics.evaluate(
                    example_id=example.id,
                    generated_answer=output['answer'],
                    ground_truth_answer=example.ground_truth_answer,
                    retrieved_files=output['retrieved_files'],
                    ground_truth_files=example.ground_truth_files,
                    ground_truth_keywords=example.ground_truth_keywords,
                    tokens_used=output['tokens_used'],
                    baseline_tokens=BASELINE_TOKENS,
                    generation_time=output['generation_time'],
                    num_retrievals=output['num_retrievals'],
                    method=f'threshold_{threshold}',
                )
                results_exp2[threshold].append(eval_result)
                
            except Exception as e:
                print(f"  Error on {example.id}: {e}")
        
        if results_exp2[threshold]:
            avg_correct = np.mean([r.answer_correctness for r in results_exp2[threshold]])
            avg_retrievals = np.mean([r.num_retrievals for r in results_exp2[threshold]])
            print(f"  Avg correctness: {avg_correct:.3f}, Avg retrievals: {avg_retrievals:.1f}")
    
    # Reset to default threshold
    adaptive_generator.config.uncertainty_threshold = 2.5
    adaptive_generator._init_monitor()
    
    print("\nExperiment 2 Complete!")
else:
    print("Skipping - no adaptive generator available")

In [ ]:
# Cell 19: Plot Threshold Sensitivity
if results_exp2:
    exp2_data = []
    for threshold, results in results_exp2.items():
        if results:
            exp2_data.append({
                'Threshold': threshold,
                'Correctness': np.mean([r.answer_correctness for r in results]),
                'Completeness': np.mean([r.answer_completeness for r in results]),
                'Efficiency': np.mean([r.token_efficiency for r in results]),
                'Num Retrievals': np.mean([r.num_retrievals for r in results]),
            })
    
    df_exp2 = pd.DataFrame(exp2_data)
    print("Threshold Sensitivity Results")
    print(df_exp2.to_string(index=False))
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    ax1 = axes[0]
    ax1.plot(df_exp2['Threshold'], df_exp2['Correctness'], 'o-', label='Correctness', color='blue', linewidth=2)
    ax1.plot(df_exp2['Threshold'], df_exp2['Completeness'], 's-', label='Completeness', color='green', linewidth=2)
    ax1.set_xlabel('CCE Threshold')
    ax1.set_ylabel('Score')
    ax1.set_title('Quality vs Threshold')
    ax1.legend()
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[1]
    ax2.plot(df_exp2['Threshold'], df_exp2['Efficiency'], 'o-', color='orange', linewidth=2)
    ax2.set_xlabel('CCE Threshold')
    ax2.set_ylabel('Token Efficiency')
    ax2.set_title('Efficiency vs Threshold')
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)
    
    ax3 = axes[2]
    ax3.plot(df_exp2['Threshold'], df_exp2['Num Retrievals'], 'o-', color='purple', linewidth=2)
    ax3.set_xlabel('CCE Threshold')
    ax3.set_ylabel('Avg Retrievals')
    ax3.set_title('Retrieval Count vs Threshold')
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/content/results/figures/exp2_threshold_sensitivity.png', dpi=150)
    plt.show()
    
    print("\nFigure saved: /content/results/figures/exp2_threshold_sensitivity.png")
else:
    print("No results to plot")

---
## 5. Save All Results

In [ ]:
# Cell 20: Save all results to JSON
from datetime import datetime

# Compile all results
all_results = {
    'timestamp': datetime.now().isoformat(),
    'config': {
        'max_tokens': MAX_TOKENS,
        'baseline_tokens': BASELINE_TOKENS,
        'methods': METHODS,
    },
    'experiment_1': {
        'name': 'CCE vs All Baselines',
        'aggregates': {k: {kk: float(vv) if isinstance(vv, (np.floating, float)) else vv 
                          for kk, vv in v.items()} 
                      for k, v in exp1_aggregates.items()},
    },
    'experiment_2': {
        'name': 'Threshold Sensitivity',
        'thresholds': THRESHOLDS,
        'data': [{k: float(v) if isinstance(v, (np.floating, float)) else v 
                  for k, v in d.items()} 
                 for d in exp2_data] if 'exp2_data' in dir() else [],
    },
}

# Save to JSON
with open('/content/results/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print("Results saved to /content/results/")
print("\nFiles:")
!ls -la /content/results/
!ls -la /content/results/figures/

In [ ]:
# Cell 21: Generate LaTeX Table for Paper
if exp1_aggregates:
    print("\nLaTeX Table for Paper")
    print("="*60)
    
    latex_table = r"""\begin{table}[h]
\centering
\caption{Comparison of CCE Adaptive with Baseline Methods}
\label{tab:main_results}
\begin{tabular}{lcccc}
\toprule
Method & Correctness & Context F1 & Efficiency & Composite \\
\midrule
"""
    
    for method in METHODS:
        if method in exp1_aggregates:
            agg = exp1_aggregates[method]
            latex_table += f"{method.replace('_', '\\_')} & {agg['answer_correctness']:.3f} & {agg['context_f1']:.3f} & {agg['token_efficiency']:.3f} & {agg['composite_score']:.3f} \\\\\n"
    
    latex_table += r"""\bottomrule
\end{tabular}
\end{table}
"""
    
    print(latex_table)
    
    # Save
    with open('/content/results/tables/main_results.tex', 'w') as f:
        f.write(latex_table)
    
    print("\nSaved to /content/results/tables/main_results.tex")

In [ ]:
# Cell 22: Final Summary and Download
print("="*70)
print("EXPERIMENTS COMPLETE")
print("="*70)

print(f"""
Summary:

Experiment 1: CCE Adaptive vs All Baselines
  - Methods tested: {len(exp1_aggregates)}
  - Examples evaluated: {exp1_aggregates.get('cce_adaptive', {}).get('n', 0) if exp1_aggregates else 0}
  - Best method: {max(exp1_aggregates.items(), key=lambda x: x[1]['composite_score'])[0] if exp1_aggregates else 'N/A'}

Experiment 2: Threshold Sensitivity
  - Thresholds tested: {THRESHOLDS}
  - Best threshold: {df_exp2.loc[df_exp2['Correctness'].idxmax(), 'Threshold'] if 'df_exp2' in dir() and len(df_exp2) > 0 else 'N/A'}

Output Files:
  - results/all_results.json
  - results/exp1_method_comparison.csv
  - results/figures/exp1_method_comparison.png
  - results/figures/exp2_threshold_sensitivity.png
  - results/tables/main_results.tex
""")

# Download all results
try:
    from google.colab import files
    import shutil
    
    # Zip results
    shutil.make_archive('/content/week9_10_results', 'zip', '/content/results')
    files.download('/content/week9_10_results.zip')
except:
    print("\nResults available in /content/results/")